# Normal Inference: Body Temperature

Each test is an independent editable block. Change the null value, alternative, and alpha, then rerun the block.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))
import pandas as pd
from classical_toolbox import ClassicalToolbox

# Dataset choice
# Keep True for the included default dataset.
# Set to False to use your own dataset, then enter the file name WITH extension.
# Put custom files either in the project folder or in the data/ folder.
use_default_dataset = True
custom_file_name = "your_classical_data.csv"  # examples: my_data.csv, my_data.xlsx, my_data.json

def choose_data_file(default_file_name, custom_file_name, use_default_dataset):
    if use_default_dataset:
        return ROOT / 'data' / default_file_name
    if not custom_file_name or custom_file_name == "your_classical_data.csv":
        raise ValueError("Set custom_file_name to your file name with extension, for example 'my_data.csv'.")
    custom_path = Path(custom_file_name)
    if custom_path.suffix.lower() not in {'.csv', '.xlsx', '.xls', '.json'}:
        raise ValueError("Custom file must include a supported extension: .csv, .xlsx, .xls, or .json")
    candidates = [custom_path, ROOT / custom_path, ROOT / 'data' / custom_path.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {custom_file_name}. Put it in the project folder or data/ folder.")

def validate_for_classical(df, required_columns, numeric_columns, categorical_columns=None):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns for this notebook: {missing}. Available columns: {list(df.columns)}")
    for col in numeric_columns:
        numeric = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(numeric) < 2:
            raise ValueError(f"Column '{col}' must contain at least two numeric observations.")
    for col in categorical_columns or []:
        if df[col].dropna().nunique() < 2:
            raise ValueError(f"Column '{col}' must contain at least two groups/categories.")

# If using a custom dataset, update these column names and the later test blocks to match your file.
required_columns = ['temperature', 'gender']
numeric_columns = ['temperature']
categorical_columns = ['gender']

data_path = choose_data_file('body_temperature.csv', custom_file_name, use_default_dataset)
print('Loaded file:', data_path)
tool = ClassicalToolbox(data_path)
df = tool.data
validate_for_classical(df, required_columns, numeric_columns, categorical_columns)
display(df.head())
print('Shape:', df.shape)
display(df.isna().sum().to_frame('missing'))
display(tool.summarize_data())

## Block 1: One-sample mean test
Question: is the average body temperature different from a hypothesized normal value?

In [ ]:
column = 'temperature'
null_value = 98.6
alternative = 'two-sided'  # 'two-sided', 'greater', or 'less'
alpha = 0.05

result = tool.one_sample_mean_test(column, null_value, alternative=alternative, alpha=alpha)
print('H0: mu =', null_value)
print('H1:', alternative)
display(tool.format_result(result))
tool.plot_test_distribution(result)
tool.plot_confidence_interval(result);

## Block 2: Two-sample mean test
Question: is mean temperature different between genders?

In [ ]:
group_col = 'gender'
value_col = 'temperature'
group_a = 'F'
group_b = 'M'
null_difference = 0
alternative = 'two-sided'
alpha = 0.05

result = tool.two_sample_mean_test(group_col, value_col, group_a, group_b, null_difference, alternative, alpha)
print(f'H0: mean({group_a}) - mean({group_b}) = {null_difference}')
print('H1:', alternative)
display(tool.format_result(result))
tool.plot_test_distribution(result)
tool.plot_confidence_interval(result);

## Block 3: One-sample variance test

In [ ]:
column = 'temperature'
null_variance = 0.7**2
alternative = 'two-sided'
alpha = 0.05

result = tool.one_sample_variance_test(column, null_variance, alternative, alpha)
print('H0: sigma^2 =', null_variance)
print('H1:', alternative)
display(tool.format_result(result))
tool.plot_test_distribution(result)
tool.plot_confidence_interval(result);

## Block 4: One-sample proportion test
Create a binary variable for temperatures above 98.6°F and test the success probability.

In [ ]:
df2 = df.copy()
df2['above_986'] = (df2['temperature'] > 98.6).astype(int)
prop_tool = ClassicalToolbox(df2)

success_col = 'above_986'
success_value = 1
null_probability = 0.5
alternative = 'less'
alpha = 0.05

result = prop_tool.one_sample_proportion_test(success_col, success_value, null_probability, alternative, alpha)
print('H0: p =', null_probability)
print('H1:', alternative)
display(prop_tool.format_result(result))
prop_tool.plot_test_distribution(result)
prop_tool.plot_confidence_interval(result);